In [10]:
import RNA
import numpy as np
from tqdm import tqdm
from Bio import SeqIO as io
import pandas as pd

In [121]:
recs = []
#Parsing data
for rec in tqdm(io.parse("../rfam.fasta", "fasta")):
    headers = rec.description.split(maxsplit=1)
    seqid = headers[0]
    desc = headers[1] if len(headers) > 1 else ""
    seq = rec.seq

    if 1:#len(seq) > 299 and len(seq) < 1001: REINTRODUCE CONDITION!
        recs.append({
            "sequence_id": seqid,#.split("_")[0],
            "description": desc,
            "sequence": str(seq)
        })
records = pd.DataFrame(recs)

7818286it [00:25, 304839.28it/s]


In [77]:
matrix = pd.read_csv("../alignment_matrix_3_perfam.csv", index_col=0) #reading matrix indices because I forgot to set the random seed when making it
ids = matrix.index.tolist()
ids.sort()
sub3pf = records.iloc[ids]

In [73]:
seqs = []
#Appending sequences
for seq in tqdm(io.parse("../rfam_head.fasta", "fasta")):
    if 1:#len(seq) > 299 and len(seq) < 1001: REINTRODUCE CONDITION!
        seqs.append(str(seq.seq))

9it [00:00, 22933.62it/s]


In [55]:
def min_free_energy(seq):
    '''
    calculates a fold and minimum free energy given a sequence
    '''
    
    fc = RNA.fold_compound(seq) #Creating 2D fold
    (ss, mfe) = fc.mfe() #Calculating the minimum free energy for the fold
    
    return(mfe)

In [59]:
def pairing_probability(seq):
    '''
    calculates the base pairing propensity per nucleotide
    '''
    
    fc = RNA.fold_compound(seq) #Create 2D fold

    (propensity, ensemble_energy) = fc.pf() #Calculating propensity and ensemble energy for the next step
    
    bpp = np.asarray(fc.bpp()) #Calculating base pairing probabilities
    bpp = np.delete(np.delete(bpp, 0, 1), 0, 0) #Trimming first row and column

    bpp = bpp + bpp.transpose() #Making the matrix symmetric (i.e. the probability of base a bonding to b is the same as b bonding to a)

    assert(bpp.shape[0] == bpp.shape[1] and bpp.shape[0] == len(seq)) #Just as a little safety measure for now
    
    base_bpp = np.sum(bpp, 0)
    return(base_bpp)

In [175]:
mfes = [] #minimum free energies matrix
bpps = [] #base pairing probability matrix (rows are individual RNAs, where nucleotide bpps are separated by |) 
for di in tqdm(ids):
    seq = records.iloc[di]["sequence"]
    mfes.append([min_free_energy(seq)])
    bpps.append(['|'.join(map(str, pairing_probability(seq)))])

100%|█████████████████████████████████████████| 513/513 [00:18<00:00, 28.08it/s]


In [183]:
#Exporting data
df_mfes = pd.DataFrame(mfes, columns = ["min_free_energy"])
df_mfes = df_mfes.set_axis(ids)
df_mfes.to_csv("mfes_3_perfam.csv")
df_bpps = pd.DataFrame(bpps, columns = ["base_pairing_probabilities"])
df_bpps = df_bpps.set_axis(ids)
df_bpps.to_csv("bpps_3_perfam.csv")